In [1]:
import os
import sys
import time
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from langchain_groq import ChatGroq

root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.bm25 import BM25Search
from src.semantic import SemanticSearch
from src.hybrid import HybridRetriever
from src.rag_pipeline import (
    load_hybrid_retriever,
    relevant_text_hybrid,
    answer_query_hybrid,
    clean_response
)

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [2]:
# initiate LLM endpoint and model
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.2, #adjust higher for more creative responses,
    max_tokens=3000,
    )

response = llm.invoke("What is a washing machine?")
print(clean_response(response.content))

A **washing machine** is a household appliance designed to automate the process of cleaning clothes, linens, and other fabrics. It replaces the manual labor of scrubbing, rinsing, and wringing out garments by using water, detergent, and mechanical action to remove dirt and stains. Here's a breakdown of its key aspects:

---

### **How It Works**  
1. **Loading**: Clothes are placed into a rotating drum.  
2. **Water and Detergent**: The machine fills with water and adds detergent (often via a dispenser).  
3. **Agitation/Spin**:  
   - **Top-loading** machines use an agitator (a central post) to move clothes through water.  
   - **Front-loading** machines rotate the drum to tumble clothes gently.  
4. **Draining and Spinning**: After washing, water is drained, and the drum spins rapidly to remove excess moisture.  

---

### **Key Components**  
- **Drum**: Holds clothes during the wash cycle.  
- **Water Inlet/Outlet**: Controls water flow in and out.  
- **Control Panel**: Allows us

In [3]:
# load processed data and indices for retrieval
data_path = root_dir / "data" / "processed" / "processed.parquet"
df = pd.read_parquet(data_path)
df_by_asin = df.set_index("parent_asin", drop=False)

print(f"Loaded {len(df)} products")

Loaded 20000 products


In [4]:
# load hybrid retriever
hybrid_retriever = load_hybrid_retriever()

print("Hybrid retriever loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hybrid retriever loaded successfully


In [5]:
# test hybrid retrieval alone without LLM
query = "energy efficient washing machine with quiet operation"
results = hybrid_retriever.retrieve(query, top_k=5)

print(f"Query: '{query}'")
print(f"\nTop {len(results)} results:")
for rank, (asin, score) in enumerate(results, start=1):
    title = df_by_asin.loc[asin].get("product_title", "Unknown Title") if asin in df_by_asin.index else "Unknown Title"
    print(f" {rank}. [{asin}] {title[:60]} | RRF Score: {score:.4f}")

Query: 'energy efficient washing machine with quiet operation'

Top 5 results:
 1. [B08R74X48Q] HOTSTORE Mini Washing Machine W/Spin Dryer, Electric Compact | RRF Score: 0.0137
 2. [B00MOD3T9A] General Electric WH20X10019 Washing Machine Drive Motor | RRF Score: 0.0136
 3. [B08XZJDMP4] Oneconcept Ecowash Pico – Portable Washing Machine, Compact  | RRF Score: 0.0083
 4. [B08F4YGV8R] [SOPHIE MCCARTHY] 2020 NEW Versions Ultrasonic Turbine Steri | RRF Score: 0.0083
 5. [B0B4P6G5QR] Portable Clothes Washing Machine,Ozone Sterilization Mini Wa | RRF Score: 0.0082


In [6]:
# test hybrid RAG pipeline with LLM response generation
answer, results = answer_query_hybrid(
    query,
    df_by_asin=df_by_asin,
    hybrid_retriever=hybrid_retriever,
    llm=llm,
)

print(f"Query: '{query}'")
print(f"\nAnswer:\n{answer}")

Query: 'energy efficient washing machine with quiet operation'

Answer:
The **Portable Clothes Washing Machine (ASIN: B0B4P6G5QR)** claims "Quick And Quiet Operation" and is priced at $49.99. While energy efficiency isn't explicitly stated, its compact design and ozone sterilization suggest water and energy savings. For energy efficiency with confirmed quiet operation, the **General Electric WH20X10019 Drive Motor (ASIN: B00MOD3T9A)** is rated 5.0 (1 review) and noted to "humms quietly," though it’s a replacement part, not a full washer.


In [7]:
# define evaluation queries (reuse from milestone 1)
evaluation_queries = [
"Washing machine",
"Stainless steel coffee maker",
"Replacement Parts DC61-02610A",
"Magic bullet",
"Whirlpool",
"Washingmachine",
"kitchen device to heat food quickly",
"Something to cook pizza in",
"Best appliances for a small apartment",
"Aquamarine appliance to make bread crispy",
]

In [8]:
# run all evaluation queries through hybrid RAG pipeline and print results
for i, q in enumerate (evaluation_queries, start=1):
    print(f"{'='*60}")
    print(f"Query {i}: '{q}'")
    print(f"{'='*60}")

    answer, results = answer_query_hybrid(
        q,
        df_by_asin=df_by_asin,
        hybrid_retriever=hybrid_retriever,
        llm=llm,
    )

    print(f"\nAnswer:\n{answer}")
    print(f"\nTop retrieved products:")
    for rank, (asin, score) in enumerate(results, start=1):
        title = df_by_asin.loc[asin].get("product_title", "Unknown Title") if asin in df_by_asin.index else "Unknown Title"
        print(f" {rank}. {title[:60]} | RRF Score: {score:.4f}")
    print()

    time.sleep(15) # add delay to avoid hitting rate limits

Query 1: 'Washing machine'

Answer:
Based on the Amazon datasets, here are key insights about portable washing machines:

1. **Formemory Mini Washing Machine (B08FMND788)**  
   - **Rating**: 5.0 (1 review, 2 helpful votes)  
   - **Pros**: Collapsible design, ideal for small/light loads (e.g., dog clothing).  
   - **Note**: No price listed, but highly praised for portability.  

2. **DACHUANG Ultrasonic Washing Machine (B099NDZ8WC)**  
   - **Rating**: 5.0 (1 review, 0 helpful votes)  
   - **Pros**: Effective cleaning, USB-powered, folds for storage.  
   - **Cons**: Lacks a wringing function.  

3. **Midea Portable Top-Loader (B01N0OI4YQ)**  
   - **Rating**: 4.0 (1 review, 2 helpful votes)  
   - **Cons**: Clothes (pants/shirts) twist together during cycles.  

For budget-conscious buyers, the **Pink Mini USB Washing Machine (B09L1LHFP8)** is priced at $17.89 but has no reviews.  

**Top recommendation**: The Formemory (B08FMND788) and DACHUANG (B099NDZ8WC) models are best for sma